# 🏭 EchoFactory: Multi-SNR Acoustic AI Training Notebook (-6 dB, 0 dB, 6 dB)
### Arsitektur: STgram-MFN v3 (Dual-Branch MobileFaceNet + ArcFace Metric Learning)
**COMPFEST 18 AI Innovation Challenge | Smart Manufacturing**

---

## 📌 Fitur & Keunggulan Notebook Ini:
1. **Multi-SNR Support**: Mendukung training pada kondisi kebisingan **-6 dB** (kebisingan pabrik ekstrem), **6 dB** (kebisingan rendah), **0 dB** (standar benchmark), serta **Mixed Multi-SNR** (Noise-Robust Acoustic Model).
2. **Google Drive & Kaggle Auto-Detection**: Otomatis mendeteksi lingkungan Google Colab (Mount Google Drive), Kaggle (`/kaggle/input`), atau Local Directory.
3. **In-Memory RAM Preloading**: Memuat seluruh spektrogram Log-Mel & High-Res Linear STFT ke RAM di awal (~2-3 detik/epoch, total training ~3-5 menit di GPU T4/V100).
4. **Deep Metric Learning (ArcFace $s=30, m=0.5$)**: Menghasilkan pemisahan embedding inter-class yang tajam antar ID mesin normal dan anomali.
5. **Per-ID Anomaly Scoring**: Evaluasi presisi berbasis KNN-Cosine k=1..5, ArcFace Target Class Distance, OCSVM, dan Rank Ensemble.
6. **Export Siap Pakai**: Otomatis mengekspor model `.pt`, model ultra-ringan `.onnx` (183 KB), dan file konfigurasi `inference_config.json`.

In [ ]:
# =====================================================================
# CELL 1: SETUP ENVIRONMENT & AUTO-MOUNT GOOGLE DRIVE / KAGGLE
# =====================================================================
import os, sys, gc, glob, json, math, time, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
from tqdm.auto import tqdm

# 1. Deteksi Lingkungan Eksekusi
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('🚀 Terdeteksi lingkungan: Google Colab')
    from google.colab import drive
    drive.mount('/content/drive')
    # Sesuaikan path folder dataset di Google Drive Anda:
    DEFAULT_DATASET_ROOT = '/content/drive/MyDrive/COMPFEST/dataset_mimii'
    DEFAULT_OUT_DIR = '/content/drive/MyDrive/COMPFEST/output_models'
elif IS_KAGGLE:
    print('🚀 Terdeteksi lingkungan: Kaggle')
    DEFAULT_DATASET_ROOT = '/kaggle/input/datasets/bisheshgiri/mimii-dataset'
    DEFAULT_OUT_DIR = '/kaggle/working'
else:
    print('🚀 Terdeteksi lingkungan: Local Machine')
    DEFAULT_DATASET_ROOT = './dataset_mimii'
    DEFAULT_OUT_DIR = './output_models'

os.makedirs(DEFAULT_OUT_DIR, exist_ok=True)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

print(f'\nDevice Komputasi : {device}')
if torch.cuda.is_available():
    print(f'GPU Name         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Tersedia    : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')


In [ ]:
# =====================================================================
# CELL 2: KONFIGURASI TARGET SNR & MESIN
# =====================================================================
# Pilihan TARGET_SNR:
#   '-6_dB'  : Kondisi kebisingan pabrik sangat ekstrem / tinggi
#   '6_dB'   : Kondisi kebisingan rendah / clean factory floor
#   '0_dB'   : Kondisi standar benchmark IEEE
#   'ALL'    : Melatih gabungan seluruh SNR (-6dB, 0dB, 6dB) untuk robustness maksimal
TARGET_SNR   = '-6_dB'  # <-- GANTI KE '6_dB', '-6_dB', '0_dB', ATAU 'ALL'

# Pilihan MACHINE_TYPE: 'fan', 'pump', 'slider', 'valve'
MACHINE_TYPE = 'fan'    # <-- GANTI SESUAI MESIN TARGET

# Path dataset & output (Ubah jika folder di Drive Anda berbeda)
DATASET_ROOT = DEFAULT_DATASET_ROOT
OUT_DIR      = DEFAULT_OUT_DIR

# Parameter Audio & Spektral
SR        = 16000
AUDIO_LEN = SR * 10  # 10 detik = 160.000 sampel PCM mono

# Parameter STgram (Branch 0: Log-Mel | Branch 1: High-Res Linear STFT)
N_MELS    = 128
N_FFT_MEL = 1024
HOP_MEL   = 512

N_FFT_TG  = 512
HOP_TG    = 256
N_BINS_TG = 128

# Hyperparameter Model & Training
EMBED_DIM    = 128
ARC_S        = 30.0   # ArcFace Scale parameter
ARC_M        = 0.5    # ArcFace Angular Margin (radian)
BATCH_SIZE   = 64
EPOCHS       = 100
LR           = 5e-4
WARMUP_EP    = 15
WEIGHT_DECAY = 1e-3

print('=' * 65)
print(f'Target Mesin      : {MACHINE_TYPE.upper()}')
print(f'Target SNR        : {TARGET_SNR}')
print(f'Dataset Path      : {DATASET_ROOT}')
print(f'Output Directory  : {OUT_DIR}')
print(f'Hyperparameters   : Epochs={EPOCHS} | Batch Size={BATCH_SIZE} | LR={LR}')
print('=' * 65)


In [ ]:
# =====================================================================
# CELL 3: SPECAUGMENT REGULARIZER
# =====================================================================
class SpecAugment(nn.Module):
    """
    Regularisasi Frequency & Time Masking pada spektrogram.
    Mencegah model overfit terhadap noise statis pada SNR ekstrem (-6dB & 6dB).
    """
    def __init__(self, freq_mask=12, time_mask=16):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask

    def forward(self, x):
        if not self.training:
            return x
        B, C, F_dim, T_dim = x.shape
        out = x.clone()
        for b in range(B):
            # Frequency Masking
            f_len = torch.randint(0, self.freq_mask + 1, (1,)).item()
            if f_len > 0 and F_dim > f_len:
                f_0 = torch.randint(0, F_dim - f_len, (1,)).item()
                out[b, :, f_0:f_0+f_len, :] = 0
            # Time Masking
            t_len = torch.randint(0, self.time_mask + 1, (1,)).item()
            if t_len > 0 and T_dim > t_len:
                t_0 = torch.randint(0, T_dim - t_len, (1,)).item()
                out[b, :, :, t_0:t_0+t_len] = 0
        return out

print('✅ SpecAugment module initialized.')


In [ ]:
# =====================================================================
# CELL 4: ARSITEKTUR MODEL STgram-MFN v3 + ArcFace Loss
# =====================================================================
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False),
            nn.BatchNorm2d(oc),
            nn.PReLU(oc)
        )
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNPReLU(ic, ic, s=s, g=ic),
            ConvBNPReLU(ic, oc, k=1, p=0)
        )
    def forward(self, x): return self.net(x)

class MobileFaceNetEncoder(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.aug = SpecAugment(freq_mask=12, time_mask=16)
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2),
            DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2),
            DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2),
            DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, ed),
            nn.BatchNorm1d(ed)
        )
    def forward(self, x):
        x = self.aug(x)
        return self.head(self.enc(x))

class ArcFaceLoss(nn.Module):
    def __init__(self, ed, nc, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, feat, labels):
        cos = F.normalize(feat, dim=1) @ F.normalize(self.W, dim=1).T
        sin = (1.0 - cos.pow(2)).clamp(1e-9).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        oh  = F.one_hot(labels, cos.shape[1]).float()
        return F.cross_entropy((oh * phi + (1.0 - oh) * cos) * self.s, labels)

    @torch.no_grad()
    def class_weights(self):
        return F.normalize(self.W, dim=1)

class STgramMFN_v3(nn.Module):
    def __init__(self, n_classes, ed=128):
        super().__init__()
        self.mel_encoder   = MobileFaceNetEncoder(ed)
        self.tgram_encoder = MobileFaceNetEncoder(ed)
        self.fuse = nn.Sequential(
            nn.Linear(ed * 2, ed),
            nn.BatchNorm1d(ed),
            nn.PReLU(ed)
        )
        self.arc = ArcFaceLoss(ed, n_classes, s=ARC_S, m=ARC_M)

    def forward(self, mel, tg, labels=None):
        f_mel = self.mel_encoder(mel)
        f_tg  = self.tgram_encoder(tg)
        feat  = F.normalize(self.fuse(torch.cat([f_mel, f_tg], dim=1)), dim=1)

        if labels is not None:
            return feat, self.arc(feat, labels)
        return feat

# Test Model Instantiation
dummy_model = STgramMFN_v3(n_classes=4, ed=EMBED_DIM)
n_p = sum(p.numel() for p in dummy_model.parameters() if p.requires_grad)
print(f'✅ Arsitektur STgram-MFN v3 Siap | Total Parameter: {n_p:,} ({n_p/1e6:.2f}M)')
del dummy_model


In [ ]:
# =====================================================================
# CELL 5: DATASET PRELOADER (Mendukung Single SNR atau Multi-SNR Mixed)
# =====================================================================
class MultiSNR_MIMIIDataset_RAM(Dataset):
    """
    Memuat spektrogram Log-Mel & Tgram ke RAM untuk training ultra-cepat.
    Mendukung SNR spesifik (-6_dB, 6_dB, 0_dB) atau seluruh SNR sekaligus.
    """
    def __init__(self, root, machine, target_snr, cond='normal', sr=16000, audio_len=160000):
        self.sr = sr
        self.audio_len = audio_len
        
        # Tentukan daftar folder SNR
        if target_snr == 'ALL':
            snr_list = ['-6_dB', '0_dB', '6_dB']
        elif isinstance(target_snr, list):
            snr_list = target_snr
        else:
            snr_list = [target_snr]
            
        # Kumpulkan semua file audio
        file_entries = []
        all_ids = set()
        
        for snr in snr_list:
            # Cari struktur path fleksibel (root/snr_machine/machine atau root/machine/id)
            candidate_paths = [
                os.path.join(root, f'{snr}_{machine}', machine),
                os.path.join(root, f'{snr}_{machine}'),
                os.path.join(root, machine),
                root
            ]
            
            found_path = None
            for cp in candidate_paths:
                if os.path.exists(cp) and any(os.path.isdir(os.path.join(cp, d)) for d in os.listdir(cp) if 'id_' in d.lower()):
                    found_path = cp
                    break
                    
            if not found_path:
                print(f'⚠️ [Info] Path untuk {snr} {machine} tidak ditemukan di kandidat umum. Mencari via glob...')
                glob_search = glob.glob(os.path.join(root, f'*{snr}*{machine}*', '**/id_*', cond, '*.wav'), recursive=True)
                for fp in glob_search:
                    parts = os.path.normpath(fp).split(os.sep)
                    mid = [p for p in parts if 'id_' in p.lower()][-1]
                    all_ids.add(mid)
                    file_entries.append((fp, mid, snr))
                continue
                
            ids = sorted([d for d in os.listdir(found_path) if os.path.isdir(os.path.join(found_path, d)) and 'id_' in d.lower()])
            for mid in ids:
                all_ids.add(mid)
                cond_dir = os.path.join(found_path, mid, cond)
                if os.path.exists(cond_dir):
                    wavs = sorted(glob.glob(os.path.join(cond_dir, '*.wav')))
                    for w in wavs:
                        file_entries.append((w, mid, snr))
                        
        if len(file_entries) == 0:
            raise FileNotFoundError(f'Tidak ada file audio ditemukan untuk {machine} ({target_snr}, {cond}) di {root}. Cek kembali path folder.')
            
        self.unique_ids = sorted(list(all_ids))
        self.id2label = {mid: i for i, mid in enumerate(self.unique_ids)}
        self.n_classes = len(self.unique_ids)
        
        print(f'\n[Dataset: {machine.upper()} | SNR: {target_snr} | Kondisi: {cond}]')
        print(f'  Total File Ditemukan : {len(file_entries)}')
        print(f'  Daftar Machine IDs   : {self.id2label}')
        print(f'  Preloading ke RAM...')
        
        self.mels = []
        self.tgrams = []
        self.labels = []
        self.mids = []
        
        for fpath, mid, snr in tqdm(file_entries, desc=f'Preload {cond} ({target_snr})'):
            wav, _ = sf.read(fpath, dtype='float32')
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            if len(wav) >= self.audio_len:
                wav = wav[:self.audio_len]
            else:
                wav = np.pad(wav, (0, self.audio_len - len(wav)))
                
            # Branch 0: Mel-Spectrogram
            mel = librosa.feature.melspectrogram(y=wav, sr=self.sr, n_mels=N_MELS, n_fft=N_FFT_MEL, hop_length=HOP_MEL)
            mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
            
            # Branch 1: High-Res Linear STFT (T-gram)
            stft = np.abs(librosa.stft(y=wav, n_fft=N_FFT_TG, hop_length=HOP_TG))
            tg = stft[:N_BINS_TG, :]
            tg_db = librosa.amplitude_to_db(tg, ref=np.max).astype(np.float32)
            
            # Bilinear Resample ke (128, 128)
            t_mel = torch.from_numpy(mel_db).unsqueeze(0).unsqueeze(0)
            t_tg  = torch.from_numpy(tg_db).unsqueeze(0).unsqueeze(0)
            t_mel = F.interpolate(t_mel, (128, 128), mode='bilinear', align_corners=False).squeeze(0)
            t_tg  = F.interpolate(t_tg, (128, 128), mode='bilinear', align_corners=False).squeeze(0)
            
            self.mels.append(t_mel)
            self.tgrams.append(t_tg)
            self.labels.append(self.id2label[mid])
            self.mids.append(mid)
            
        self.mels_tensor = torch.stack(self.mels)
        self.tgrams_tensor = torch.stack(self.tgrams)
        self.labels_tensor = torch.tensor(self.labels, dtype=torch.long)
        print(f'✅ Berhasil preload {len(self.labels)} sampel ({cond}) ke RAM.')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.mels_tensor[idx], self.tgrams_tensor[idx], self.labels_tensor[idx]

# Inisialisasi Dataset Training (Kondisi Normal)
train_ds = MultiSNR_MIMIIDataset_RAM(
    root=DATASET_ROOT, machine=MACHINE_TYPE, target_snr=TARGET_SNR, cond='normal',
    sr=SR, audio_len=AUDIO_LEN
)
N_CLASSES = train_ds.n_classes
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
print(f'\nDataLoader Siap: {len(train_dl)} batches/epoch | {N_CLASSES} Machine IDs | Batch Size={BATCH_SIZE}')


In [ ]:
# =====================================================================
# CELL 6: TRAINING LOOP — ArcFace Metric Learning & Cosine Annealing
# =====================================================================
model = STgramMFN_v3(n_classes=N_CLASSES, ed=EMBED_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

def get_lr(ep):
    if ep <= WARMUP_EP:
        return LR * ep / WARMUP_EP
    p = (ep - WARMUP_EP) / (EPOCHS - WARMUP_EP)
    return LR * 0.5 * (1 + math.cos(math.pi * p))

OUT_MODEL_PATH = os.path.join(OUT_DIR, f'stgram_mfn_v3_{MACHINE_TYPE}_{TARGET_SNR}.pt')

print(f'\nMemulai Training STgram-MFN v3...')
print(f'  Target SNR : {TARGET_SNR}')
print(f'  Epochs     : {EPOCHS} | Peak LR: {LR}')
print('=' * 65)

best_loss = float('inf')
losses = []
t0 = time.time()

for ep in range(1, EPOCHS + 1):
    lr_now = get_lr(ep)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    model.train()
    ep_loss = 0.0
    n_bat = 0

    for mel, tg, lab in train_dl:
        mel = mel.to(device, non_blocking=True)
        tg  = tg.to(device, non_blocking=True)
        lab = lab.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            _, loss = model(mel, tg, lab)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        ep_loss += loss.item()
        n_bat += 1

    avg_loss = ep_loss / max(n_bat, 1)
    losses.append(avg_loss)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': ep,
            'model_state': model.state_dict(),
            'best_loss': best_loss,
            'machine': MACHINE_TYPE,
            'target_snr': TARGET_SNR,
            'n_classes': N_CLASSES,
            'embed_dim': EMBED_DIM,
            'id2label': train_ds.id2label,
            'arc_W': model.arc.class_weights().cpu(),
        }, OUT_MODEL_PATH)
        tag = ' <- BEST'
    else:
        tag = ''

    if ep % 10 == 0 or ep == 1 or ep == EPOCHS:
        print(f'Epoch {ep:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {lr_now:.2e} | Waktu: {(time.time()-t0)/60:.1f} min{tag}')

print(f'\n🎉 Training Selesai! Best Loss: {best_loss:.4f} | Total Waktu: {(time.time()-t0)/60:.2f} menit')

# Visualisasi Kurva Loss
plt.figure(figsize=(9, 3.8))
plt.plot(losses, 'b-', lw=1.6, label='ArcFace Loss')
plt.axvline(x=WARMUP_EP-1, color='orange', linestyle='--', alpha=0.7, label=f'End Warmup (Ep {WARMUP_EP})')
plt.axhline(y=best_loss, color='green', linestyle='--', alpha=0.7, label=f'Best: {best_loss:.4f}')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Loss (Log Scale)'); plt.legend(); plt.grid(alpha=0.3)
plt.title(f'Training Loss Curve — {MACHINE_TYPE.upper()} ({TARGET_SNR})', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'loss_{MACHINE_TYPE}_{TARGET_SNR}.png'), dpi=150)
plt.show()


In [ ]:
# =====================================================================
# CELL 7: EVALUASI PER-ID (KNN Cosine, ArcFace Target Distance, Ensemble)
# =====================================================================
ck = torch.load(OUT_MODEL_PATH, map_location='cpu')
eval_model = STgramMFN_v3(n_classes=ck['n_classes'], ed=ck['embed_dim']).to(device)
eval_model.load_state_dict(ck['model_state'], strict=True)
eval_model.eval()

arc_W = ck['arc_W'].numpy()

# Preload Abnormal Dataset
abnorm_ds = MultiSNR_MIMIIDataset_RAM(
    root=DATASET_ROOT, machine=MACHINE_TYPE, target_snr=TARGET_SNR, cond='abnormal',
    sr=SR, audio_len=AUDIO_LEN
)

@torch.no_grad()
def extract_all_embs(model_inst, ds):
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    embs = []
    for mel, tg, _ in tqdm(dl, desc='Extract Embeddings'):
        f = model_inst(mel.to(device, non_blocking=True), tg.to(device, non_blocking=True))
        embs.append(f.cpu().numpy())
    return np.concatenate(embs, axis=0)

print('\nExtracting Normal Embeddings...')
norm_embs = extract_all_embs(eval_model, train_ds)
norm_labels = train_ds.labels_tensor.numpy()

print('Extracting Abnormal Embeddings...')
abnorm_embs = extract_all_embs(eval_model, abnorm_ds)
abnorm_labels = abnorm_ds.labels_tensor.numpy()

y_true = np.concatenate([np.zeros(len(norm_embs)), np.ones(len(abnorm_embs))])
all_labels = np.concatenate([norm_labels, abnorm_labels])
all_embs = np.concatenate([norm_embs, abnorm_embs], axis=0)

# 1. Scorer KNN Cosine Per-ID
def compute_knn_scores(k=5):
    unique_ids = np.unique(norm_labels)
    knn_models = {}
    for uid in unique_ids:
        mask = (norm_labels == uid)
        knn = NearestNeighbors(n_neighbors=min(k, mask.sum()), metric='cosine', algorithm='brute')
        knn.fit(norm_embs[mask])
        knn_models[uid] = knn
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        dists, _ = knn_models[lid].kneighbors(all_embs[i:i+1])
        scores.append(float(dists.mean()))
    return np.array(scores)

# 2. Scorer ArcFace Target Class Distance
def compute_arcface_scores():
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        w_target = arc_W[lid]
        cos_sim = np.dot(all_embs[i], w_target)
        scores.append(1.0 - cos_sim)
    return np.array(scores)

# 3. Scorer OCSVM Per-ID
def compute_ocsvm_scores(nu=0.02):
    unique_ids = np.unique(norm_labels)
    ocsvm_models, scalers = {}, {}
    for uid in unique_ids:
        mask = (norm_labels == uid)
        sc = StandardScaler(); X = sc.fit_transform(norm_embs[mask])
        oc = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
        oc.fit(X)
        ocsvm_models[uid] = oc; scalers[uid] = sc
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        X = scalers[lid].transform(all_embs[i:i+1])
        scores.append(-float(ocsvm_models[lid].decision_function(X)[0]))
    return np.array(scores)

print('\nMenghitung Skor Anomali...')
s_knn   = compute_knn_scores(k=5)
s_arc   = compute_arcface_scores()
s_ocsvm = compute_ocsvm_scores(nu=0.02)

# Rank Ensemble
r_knn   = rankdata(s_knn) / len(s_knn)
r_arc   = rankdata(s_arc) / len(s_arc)
r_ocsvm = rankdata(s_ocsvm) / len(s_ocsvm)
s_ens   = (r_knn * 0.5 + r_arc * 0.3 + r_ocsvm * 0.2)

scorers = {
    'KNN-k5 (Per-ID)': s_knn,
    'ArcFace-Target': s_arc,
    'OCSVM (Per-ID)': s_ocsvm,
    'Ensemble Scorer': s_ens
}

print('=' * 65)
print(f'HASIL EVALUASI — {MACHINE_TYPE.upper()} ({TARGET_SNR})')
print('=' * 65)
best_name, best_auc, best_pauc, best_sc = '', 0, 0, None
for name, sc in scorers.items():
    auc  = roc_auc_score(y_true, sc)
    pauc = roc_auc_score(y_true, sc, max_fpr=0.1)
    print(f'{name:20s} | AUC: {auc*100:6.2f}% | pAUC (FPR<10%): {pauc*100:6.2f}%')
    if auc > best_auc:
        best_auc, best_pauc, best_name, best_sc = auc, pauc, name, sc

fpr, tpr, thr = roc_curve(y_true, best_sc)
best_thr = float(thr[np.argmax(tpr - fpr)])

print('=' * 65)
print(f'🏆 METODE TERBAIK : [{best_name}]')
print(f'  AUC             : {best_auc*100:.2f}%')
print(f'  pAUC (FPR < 10%): {best_pauc*100:.2f}%')
print(f'  Optimal Threshold: {best_thr:.4f}')
print('=' * 65)


In [ ]:
# =====================================================================
# CELL 8: VISUALISASI HASIL BENCHMARK & DISTRIBUSI SKOR
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. ROC Curve
ax1 = axes[0]
ax1.plot(fpr, tpr, 'b-', lw=2.2, label=f'{best_name}\nAUC={best_auc:.4f}')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
mask = fpr <= 0.1
ax1.fill_between(fpr[mask], tpr[mask], alpha=0.3, color='orange', label=f'pAUC={best_pauc:.4f}')
ax1.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5)
ax1.set_title(f'ROC Curve — {MACHINE_TYPE.upper()} ({TARGET_SNR})', fontweight='bold')
ax1.set_xlabel('False Positive Rate (FPR)')
ax1.set_ylabel('True Positive Rate (TPR)')
ax1.legend(); ax1.grid(alpha=0.3)

# 2. Distribusi Skor Anomali
ax2 = axes[1]
sc_n = best_sc[y_true == 0]
sc_a = best_sc[y_true == 1]
ax2.hist(sc_n, bins=50, alpha=0.65, color='royalblue', label=f'Normal (n={len(sc_n)})', density=True)
ax2.hist(sc_a, bins=50, alpha=0.65, color='crimson', label=f'Abnormal (n={len(sc_a)})', density=True)
ax2.axvline(x=best_thr, color='green', linestyle='--', lw=2, label=f'Threshold={best_thr:.3f}')
ax2.set_title(f'Score Distribution ({best_name})', fontweight='bold')
ax2.set_xlabel('Anomaly Score')
ax2.set_ylabel('Probability Density')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'eval_{MACHINE_TYPE}_{TARGET_SNR}.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# =====================================================================
# CELL 9: EXPORT KE ONNX (183 KB) & INFERENCE CONFIG JSON
# =====================================================================
try:
    import onnx
except ImportError:
    print('Installing ONNX...')
    subprocess.run(['pip', 'install', '-q', 'onnx', 'onnxscript'], check=True)
    import onnx

m_export = STgramMFN_v3(n_classes=ck['n_classes'], ed=ck['embed_dim'])
m_export.load_state_dict(ck['model_state'], strict=True)
m_export.eval()

onnx_filename = f'stgram_mfn_v3_{MACHINE_TYPE}_{TARGET_SNR}.onnx'
onnx_filepath = os.path.join(OUT_DIR, onnx_filename)

dummy_mel = torch.randn(1, 1, 128, 128)
dummy_tg  = torch.randn(1, 1, 128, 128)

try:
    torch.onnx.export(
        m_export,
        (dummy_mel, dummy_tg),
        onnx_filepath,
        input_names=['mel', 'tgram'],
        output_names=['embedding'],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes={'mel': {0: 'B'}, 'tgram': {0: 'B'}, 'embedding': {0: 'B'}}
    )
    print(f'✅ ONNX berhasil diekspor (opset 17): {onnx_filepath} ({os.path.getsize(onnx_filepath)/1024:.1f} KB)')
except Exception as e:
    print(f'Fallback export opset 16: {e}')
    torch.onnx.export(
        m_export,
        (dummy_mel, dummy_tg),
        onnx_filepath,
        input_names=['mel', 'tgram'],
        output_names=['embedding'],
        opset_version=16,
        do_constant_folding=True,
        dynamic_axes={'mel': {0: 'B'}, 'tgram': {0: 'B'}, 'embedding': {0: 'B'}}
    )
    print(f'✅ ONNX berhasil diekspor (opset 16): {onnx_filepath} ({os.path.getsize(onnx_filepath)/1024:.1f} KB)')

# Simpan inference config JSON untuk backend / Hugging Face Spaces
cfg = {
    'version': 'v3_stgram_multidb',
    'machine': MACHINE_TYPE,
    'target_snr': TARGET_SNR,
    'best_scorer': best_name,
    'auc': float(best_auc),
    'pauc': float(best_pauc),
    'threshold': float(best_thr),
    'n_classes': int(ck['n_classes']),
    'embed_dim': int(ck['embed_dim']),
    'id2label': train_ds.id2label,
    'arc_W': arc_W.tolist(),
    'audio': {'sr': SR, 'audio_len': AUDIO_LEN},
    'features': {
        'branch_0_mel': {'n_mels': N_MELS, 'n_fft': N_FFT_MEL, 'hop_length': HOP_MEL},
        'branch_1_tgram': {'n_fft': N_FFT_TG, 'hop_length': HOP_TG, 'n_bins': N_BINS_TG}
    }
}
config_filepath = os.path.join(OUT_DIR, f'inference_config_{MACHINE_TYPE}_{TARGET_SNR}.json')
with open(config_filepath, 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'✅ Konfigurasi inferensi tersimpan: {config_filepath}')
print('\n' + '=' * 65)
print(f'🎯 TRAINING LENGKAP {MACHINE_TYPE.upper()} ({TARGET_SNR}) SELESAI!')
print(f'   AUC: {best_auc*100:.2f}% | pAUC: {best_pauc*100:.2f}% | Threshold: {best_thr:.4f}')
print('=' * 65)
